# Robot control Lab 1 - OpenCV and pixel-space basics

Today's lab will focus on basic operations with still images or, more commonly, single camera frames. We will use OpenCV ([docs](https://docs.opencv.org/4.13.0/index.html)). During creation of this scenario Google Colab runs OpenCV version 4.14; remember to use docs that match the version you are using.

## Working with Google Colab

If you are not familiar with Colab, it is a cloud-based Jupyter notebook environment, which is a virtual machine that will allow you to run your code, manage files, etc.

You can click the button below to open this notebook in Colab (if you haven't done so already); alternatively, go to https://colab.research.google.com/ and upload the file manually.

[![Open this notebook in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mim-ml-teaching/public-rc-2026-27/blob/main/docs/lab1-public/cv_notebook/lab1-colab-student.ipynb)

By default, a newly created notebook will be saved in the "Colab Notebooks" folder in your Google Drive. If you opened the notebook using the button above, you can use the "Copy to Drive" button to save it. You can use your [UW Google account](https://it.uw.edu.pl/pl/uslugi/UslugiMojaPocztaGmailStudent/) or any other account.

Go through the whole notebook by expanding subsequent sections. If you are familiar with OpenCV you can probably skip everything until "Detecting markers". And if you are already familiar with markers as well, skip to the "Extra task". In case you are skipping ahead remember to download assets and execute cells with imports.


## Working locally

Eventually you may also find that you prefer to work locally instead of Google Colab. In that case we suggest installation of [uv](https://docs.astral.sh/uv/) as a package manager; in that case set up your local python environment using `pyproject.toml` from this lab's files (`uv sync` to install the dependencies).
This notebook support both Google Colab and local environments.

## Downloading assets



Execute cell below to download assets we will be using today. If you have not launched the runtime yet it will probably launch automatically now.

In [ ]:
!rm -rf /tmp/rc-repo /tmp/lab1_assets  # clean up previously downloaded files
!git clone https://github.com/mim-ml-teaching/public-rc-2026-27 /tmp/rc-repo
!mv /tmp/rc-repo/docs/lab1-public/cv_notebook/assets /tmp/lab1_assets
!rm -rf /tmp/rc-repo  # clean-up


Here is what we now have in the `/tmp/lab1_assets` directory:

In [ ]:
!ls /tmp/lab1_assets

## OpenCV basics


Let's make sure we have the environment set up correctly first:

In [ ]:
import importlib.util


for dep in ["cv2", "matplotlib"]:
    assert importlib.util.find_spec(dep) is not None, (
        f"Looks like you haven't installed {dep} in your local environment!"
    )


First we will import OpenCV and check the version. This scenario is written when the version available in Colab by default is 4.14.

In [ ]:
import cv2
import matplotlib.pyplot as plt

print(f"OpenCV version is: {cv2.__version__}")


def cv2_imshow(img, width=None):
    """A helper which displays images in notebooks nicely"""

    if width is not None:
        dpi = 200
        h, w = img.shape[:2]
        height = width * h / w
        plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)

    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

Let's explore how to load an image file and how the variable holding the image looks like.

In [ ]:
first_img = cv2.imread("/tmp/lab1_assets/f.png")
print(type(first_img))
print(first_img)


If it looks familiar that's because the underlying type for the image is actually a NumPy array. Let's explore its shape and elements type.

In [ ]:
print(f"Shape is: {first_img.shape}")
print(f"Element type is: {first_img.dtype}")

Here is the first caveat: normally in computer graphics the first coordinate is the horizontal axis and the second one is vertical, like so:

![Image coords](https://learn365project.files.wordpress.com/2015/08/computer_coordinates.png)

But because Python's API of OpenCV uses NumPy, the image is treated as a matrix, so x and y are swapped, because that's how we usually index matrices. In this case the shape of `(822, 1024, 3)` will actually mean 822 rows, 1024 columns, so the image resolution will be 1024x822. What about that 3? This designates the number of channels, most often this means RGB channels (to learn more check [here](https://learnopencv.com/color-spaces-in-opencv-cpp-python/)).

`dtype` of `uint8` means for each pixel for each channel we have a byte (8-bit value, 0-255 range).

As a side note, when talking about image coordinates you will often see that letters `(u, v)` are used instead of `(x, y)`.

Now we are ready to preview the image:

In [ ]:
cv2_imshow(first_img)

What a nice picture!


## Resizing



Let's say we want to make the image smaller. Let's first visit the docs and find the `resize` function. That's our first function today, so [here](https://docs.opencv.org/4.13.0/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231edc7e7529d) is a little hint. Notice that the documentation is primarily for C++, but with every function you should be able to spot Python's API. So we have to pass `src` and `dsize`. Notice also that the Python version of the function returns output image both in the return value of the function and in the third parameter. Let's resize the image to 300x300 px.

In [ ]:
first_img_small_square = cv2.resize(first_img, (300, 300))
cv2_imshow(first_img_small_square)

Now it's your turn! Try to resize the image as we did above, but this time we want the height to be 300 and the width should be calculated to keep the original aspect ratio.

In [ ]:
height, width, chann = first_img.shape
dsize = (round(width/height * 300), 300)  # TODO: modify this line

first_img_small = cv2.resize(first_img, dsize)
cv2_imshow(first_img_small)

As you probably noticed, there is a difference in that `shape[0]` is y (vertical, height) and `shape[1]` is x (horizontal, width) because this is the numpy interface. When we pass something to any OpenCV function such as the `dsize` tuple here, it needs the normal order in which the first coordinate is the horizontal one.



## Cropping



We can also crop the image (cut some rectangular section). In Python, since OpenCV uses NumPy we actually do that using NumPy matrix operators (slicing). Let's say we want to cut a vertical rectangle from the middle of the original image.

In [ ]:
# "tl" is often used to designate top-left corner, "br" will be bottom-right
# We will cut a rectangle that starts in the middle of the image and goes 200 px right and 350 px down.

crop_tl_x = first_img.shape[1] // 2
crop_tl_y = first_img.shape[0] // 2
crop_width = 200
crop_height = 350
crop_br_x = crop_tl_x + crop_width
crop_br_y = crop_tl_y + crop_height

first_img_crop_0 = first_img[crop_tl_y:crop_br_y, crop_tl_x:crop_br_x]
cv2_imshow(first_img_crop_0)

Now your turn: crop a rectangle that starts in (u, v) = (175, 675) and has a vertical dimension 100 and horizontal 200.

In [ ]:
# TODO: Crop first_img_crop_1 as instructed above

first_img_crop_1 = first_img[675:775, 175: 375]
cv2_imshow(first_img_crop_1)

## Grayscale conversion



The last thing we will explore about general OpenCV tasks today will be converting an image to grayscale and then binary black and white image. You probably know OpenCV and the docs enough already to find the solution yourself (try google if you cannot find what you want in the docs).

Hint: `cv2.COLOR_BGR2GRAY`

In [ ]:
# TODO: Find a function to convert first_img (or first_img_small) to grayscale
first_img_grayscale = cv2.cvtColor(first_img, cv2.COLOR_BGR2GRAY)
cv2_imshow(first_img_grayscale)

In [ ]:
# TODO: Find a function to convert first_img_grayscale to binary and try a few different thresholds to see the effect
_, first_img_binary = cv2.threshold(first_img_grayscale, 127, 255, cv2.THRESH_BINARY)
cv2_imshow(first_img_binary)

## Detecting markers


Now let's try to do something very useful in robotics! Fiducial markers are specially designed shapes, very often squares with binary pixels that encode only the id of the marker. You can think of them as QR codes, but they are not designed as to carry a lot of information like a QR code, but rather have very distinct edge so we can detect its shape and position on the image.

![Markers](https://raw.githubusercontent.com/mim-ml-teaching/public-rc-2026-27/main/docs/lab1-public/cv_notebook/assets/alot.png)

We will take an image and run a function that detects markers. We will use `0.png` first. It is always a good idea to preview an image we are working with.

In [ ]:
# TODO: load the image named "0.png"
img0 = cv2.imread("0.png")
cv2_imshow(img0)


There are a lot of different tag families. On our images all tags are from the **AprilTag 16h5** family. We will create a dictionary object that will be passed to the detector. The detector is just a function called `detectMarkers`. Below you will find a complete call to the detector. Inspect what are the return values (or read in the docs!)

In [ ]:
dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_APRILTAG_16h5)
parameters = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(dictionary, parameters)

corners, ids, _ = detector.detectMarkers(img0)
# TODO: inspect what is returned from detectMarkers
print(corners, ids)

You can see how the output looks like for one marker in the image. We get `corners` and `ids` and every detection has 4 corners and a corresponding ID. As you can see, every corner is a pair of (x, y) coordinates. Notice the nesting of lists – you would probably expect one less level of nesting!

First, let's visualize the detections and for now let's use the default function `drawDetectedMarkers`.

Head to the docs and find out how to use it!

In [ ]:
# Hint: To not draw over original image we can create a copy
img0_draw = img0.copy()

# TODO: draw markers detected in previous step on the img0_draw
cv2.aruco.drawDetectedMarkers(img0_draw, corners, ids)
cv2_imshow(img0_draw, width=1200)

Notice that one corner of the marker is special. This will be the index corner, so that you can detect marker rotation and will always be first on the list of corners. Now your task is to crop the original image (without detections visualized) to focus on the marker. Crop the marker with some margin around all corners.

In [ ]:
# Hint: below is a list of all x coordinates of our marker's corners
xs = [corner[0] for corner in corners[0][0]]
ys = [corner[1] for corner in corners[0][0]]

# print(xs, ys)
margin = 10
left_x = int(min(xs)) - margin
right_x = int(max(xs)) + margin
left_y = int(min(ys)) - margin
right_y = int(max(ys)) + margin
# print(left_x, right_x, left_y, right_y)

img0_marker_crop = img0[left_y: right_y, left_x:right_x]
cv2_imshow(img0_marker_crop)


## Drawing markers



Now let's try to write our own `drawMarkers` function! Below is a demo of how to use drawing functions. Write your drawing function that marks sides, corners (different mark for index corner) and displays the ID.

In [ ]:
# Demo of drawing
draw_demo = first_img.copy()

RED = (0, 0, 255)
PINK = (255, 0, 255)

cv2.circle(draw_demo, (100, 200), 30, RED, cv2.FILLED)
cv2.line(draw_demo, (250, 250), (800, 100), PINK, 3)
cv2.putText(draw_demo, "(250, 250)", (250, 250), cv2.FONT_HERSHEY_SIMPLEX, 1, RED, 2)
cv2.putText(draw_demo, "(800, 100)", (800, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, RED, 2)
cv2_imshow(draw_demo)

In [ ]:
# Experiment with colors!
GREEN = (0, 255, 0)
DARK_GREEN = (0, 127, 0)
RED = (0, 0, 255)
BLUE = (255, 0, 0)
VIOLET = (255, 0, 255)
CYAN = (255, 255, 0)
YELLOW = (0, 255, 255)
ORANGE = (0, 100, 255)
margin = 10

def drawMarkers(img, corners, ids):
    if ids is None:
        return
    for marker, current_id in zip(corners, ids):
        marker = marker[0]
        prev = (int(marker[-1][0]), int(marker[-1][1]))
        for x, y in marker:
            x = int(x)
            y = int(y)
            cv2.circle(img, (x, y), 15, BLUE, cv2.FILLED)
            cv2.line(img, prev, (x, y), VIOLET, 7)
            prev = (x, y)
        left_x = int(min(corner[0] for corner in marker))
        left_y = int(min(corner[1] for corner in marker))
        cv2.putText(img, str(current_id), (left_x - margin, left_y - margin), cv2.FONT_HERSHEY_SIMPLEX, 3, RED, 2)

img0_draw = img0.copy()
drawMarkers(img0_draw, corners, ids)
cv2_imshow(img0_draw)

Remember how we called `detectMarkers` on `0.png`? Now do the same for images 1-6 (writing a function might be a good idea). Draw the results using your drawing function!

In [ ]:
# TODO: detect, draw and show markers on images 1-6.png
def get_markers(img):
  dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_APRILTAG_16h5)
  parameters = cv2.aruco.DetectorParameters()
  detector = cv2.aruco.ArucoDetector(dictionary, parameters)

  corners, ids, _ = detector.detectMarkers(img)
  return corners, ids

def detect_draw_show(filename):
  img = cv2.imread(filename)
  img_draw = img.copy()

  corners, ids = get_markers(img)

  drawMarkers(img_draw, corners, ids)
  cv2_imshow(img_draw)


for i in range(0, 8):
    detect_draw_show(str(i) + ".png")

## Extra task



If you finished early or just want a more challanging task here is one more for you. Let's take `7.png` and assume that, due to it being high resolution, running markers detection on the whole image will be very slow. Now assume you somehow know that the markers will probably be found in two areas of interest. The first area is from between `(700, 450)` and `(900, 600)` and the second area has bottom left (!) corner at `(1200, 1000)`, the height of `600` and the width of `300`. Crop the areas of interest, run markers detection and then return marker IDs with coordinates of their index corners on the original image. Call your drawing function exactly once to visualize detections. You will need combined results in the full image coordinates.

In [ ]:
img7 = cv2.imread("7.png")

img7_crop1 = img7[450:600, 700:900]
img7_crop2 = img7[400:1000, 1200:1500]

cv2_imshow(img7_crop1)
cv2_imshow(img7_crop2)

corners1, ids1 = get_markers(img7_crop1)
corners2, ids2 = get_markers(img7_crop2)

def shift_markers(corners, left_x, left_y):
    corners_shifted = []
    for marker in corners:
        new_marker = [[int(x) + left_x, int(y) + left_y] for [x, y] in marker[0]]
        corners_shifted.append([new_marker])

    return corners_shifted

corners_shifted1 = shift_markers(corners1, 700, 450)
corners_shifted2 = shift_markers(corners2, 1200, 400)
print(corners_shifted2, corners2)
corners = corners_shifted1 + corners_shifted2
ids = ids1 + ids2

img_draw = img7.copy()
drawMarkers(img_draw, corners, ids)
cv2_imshow(img_draw)
